<a href="https://colab.research.google.com/github/cassiecinzori/ECON3916/blob/main/Assignments/Assignments%204/ECON3916_Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 4: The Predictive Architecture

### Cassandra Cinzori

#### Imports

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.eval_measures import rmse
import missingno as msno
import category_encoders as ce
import matplotlib.pyplot as plt
import seaborn as sns

## Phase 1: Causal Topology and Multicollinearity Forensics

#### Step 1.1: Diagnosing Spurious Clinical Signals (DAGs)


**Observed (spurious) relationship:**
> High_Deductible_Plan -> Inpatient_Admission_Rate

**True causal structure (Fork / Common Cause):**
```
    Socioeconomic_Deprivation
           /          \
          ↓            ↓
  High_Deductible_Plan   Inpatient_Admission_Rate
```

Socioeconomic deprivation simultaneously causes patients to select high-deductible plans (lower premiums are more affordable under financial strain) and independently drives higher inpatient admission rates through deferred preventative care, chronic stress, and nutritional deficiency. The two endpoints are correlated, but neither causes the other.

Regressing admissions on insurance type without controlling for the confounder absorbs the deprivation effect directly into the insurance coefficient. The result is an inflated, causally meaningless predictive weight: a textbook case of omitted variable bias in which the model mistakes a symptom of poverty for a clinical mechanism.

#### Step 1.2: The Variance Inflation Factor (VIF) Audit


In [ ]:
vitals = pd.read_csv('OmniCare_Clinical_Vitals.csv')

# Select the continuous physiological predictors
continuous_features = ['Weight_kg', 'Height_cm', 'BMI', 'Systolic_BP', 'Diastolic_BP']

# Add a constant column — required for VIF calculation to be mathematically valid
X_vitals = sm.add_constant(vitals[continuous_features])

# Calculate VIF for each predictor (skip index 0 = constant)
vif_df = pd.DataFrame({
    'Variable': X_vitals.columns[1:],
    'VIF':      [variance_inflation_factor(X_vitals.values, i)
                 for i in range(1, X_vitals.shape[1])]
})
print("--- INITIAL VIF AUDIT ---")
print(vif_df.to_string(index=False))

# Identify and drop the most redundant variable (BMI is a direct linear
# function of Weight and Height, so it will almost always have the highest VIF)
redundant = vif_df.loc[vif_df['VIF'].idxmax(), 'Variable']
print(f"\nDropping most redundant variable: {redundant}")
continuous_features.remove(redundant)

# Recalculate VIF to confirm structural stability
X_vitals_clean = sm.add_constant(vitals[continuous_features])
vif_df_clean = pd.DataFrame({
    'Variable': X_vitals_clean.columns[1:],
    'VIF':      [variance_inflation_factor(X_vitals_clean.values, i)
                 for i in range(1, X_vitals_clean.shape[1])]
})
print("\n--- POST-CORRECTION VIF AUDIT ---")
print(vif_df_clean.to_string(index=False))

## Phase 2: Visual Forensics and The High-Cardinality Frontier

#### Step 2.1: The Architecture of Missingness

Under Donald Rubin's taxonomy, this is **Missing Not At Random (MNAR)**.

The gaps in `Continuous_Heart_Rate` are not random. They occur *because of the value of the missing data itself*. Low-income patients systematically opt out of continuous telemetry transmission due to mobile data costs. Income level is both the reason for missingness and a confound of the underlying health outcome. The data is missing precisely for the patients whose heart rate readings are most clinically significant.

**Why mean imputation destroys integrity:** Mean imputation replaces all missing values with the column average, implicitly assuming missingness is random (MCAR). Here, that assumption is false. Imputing the population mean heart rate for low-income patients would systematically replace their true (likely elevated, stress-related) readings with the healthier population average, erasing the exact signal the model needs to accurately price their risk. The imputed dataset would understate risk for the most vulnerable patients, producing both a biased model and a discriminatory pricing outcome.


In [ ]:
telemetry = pd.read_csv('OmniCare_Telemetry_Data.csv')

# missingno.matrix renders a white-stripe visualization where gaps in a column
# represent missing values. Clustered gaps = non-random (MNAR) structure.
msno.matrix(telemetry)
plt.title('Missingness Matrix — OmniCare Telemetry Data')
plt.show()

#### Step 2.2: Escaping the Dummy Variable Trap

OLS requires computing the estimator **β = (XᵀX)⁻¹Xᵀy**, which depends on the design matrix **X** being invertible.

When `pd.get_dummies()` generates all 850 ICD-10 binary columns and the model includes a constant intercept term, perfect linear dependence is introduced: the 850 dummy columns sum to exactly **1** for every observation, identical to the constant column. The design matrix therefore contains a redundant column, making it **singular** (determinant = 0) and **non-invertible**.

The OLS estimator is mathematically undefined. The system has infinitely many solutions because you cannot uniquely attribute variation in cost to any single diagnosis group when the intercept is already capturing the grand mean. The fix is either to drop one dummy (the reference category) or, far more practical at 850 categories, replace the entire column set with a single target-encoded continuous vector.

---

## Step 3.2 — RMSE: Financial & Regulatory Risk Analysis

An RMSE of **$450 on a $1,200 procedure represents a 37.5% average pricing error.** Deployed in a live hospital environment, this constitutes failure across three dimensions:

**Operational Risk**
The same MRI could be dynamically priced anywhere from ~$750 to ~$1,650 depending on model noise rather than true cost drivers. Revenue forecasting becomes unreliable, and billing reconciliation against insurance contracts breaks down.

**Financial Risk**
Systematic over-pricing exposes the hospital network to insurance clawbacks, CMS billing audits, and revenue cycle instability. Under-pricing at scale erodes margins on high-utilization procedures. Neither outcome is recoverable through model tuning alone at this error magnitude.

**Regulatory Risk**
Dynamic pricing based on patient risk profiles may violate the No Surprises Act (2022), CMS price transparency requirements, and potentially HIPAA if encoded risk scores function as proxies for protected health information. A $450 RMSE on a $1,200 procedure is not a modeling imperfection. It is a patient harm and institutional liability event. The algorithm is not ready for deployment.



#### Step 2.3: Target Encoding Implementation

In [ ]:
encoder = ce.TargetEncoder(cols=['Primary_Diagnosis_Code'])

# fit_transform learns the mapping from diagnosis → mean cost, then applies it
telemetry['Target_Encoded_Diagnosis'] = encoder.fit_transform(
    telemetry['Primary_Diagnosis_Code'],
    telemetry['Procedure_Cost_USD']
)

print("\n--- TARGET ENCODED DIAGNOSIS (first 5 rows) ---")
print(telemetry[['Primary_Diagnosis_Code', 'Target_Encoded_Diagnosis',
                  'Procedure_Cost_USD']].head())

## Phase 3: Architecting the Prediction Engine

#### Step 3.1: OLS Optimization via Patsy Formulas

In [ ]:
df_final = telemetry.merge(vitals[['Patient_ID'] + continuous_features],
                           on='Patient_ID', how='inner')

# Patsy R-style formula — no need to manually add a constant, smf.ols does it
formula = ('Procedure_Cost_USD ~ Target_Encoded_Diagnosis + '
           'Clinic_Capacity_Percentage + Time_of_Day_Index + ' +
           ' + '.join(continuous_features))

ols_model = smf.ols(formula, data=df_final).fit()
print("\n--- OLS MODEL SUMMARY ---")
print(ols_model.summary())

#### Step 3.2: Financial Loss Quantification (RMSE)

In [ ]:
fitted   = ols_model.fittedvalues
actuals  = df_final['Procedure_Cost_USD']
model_rmse = rmse(actuals, fitted)

print(f"\n--- FINANCIAL RISK QUANTIFICATION ---")
print(f"RMSE: ${model_rmse:,.2f}")
print(f"This means predictions are off by ~${model_rmse:,.2f} per procedure on average.")

#### Step 3.3: Residual Diagnostics for Heteroscedasticity

In [ ]:
residuals   = ols_model.resid
fitted_vals = ols_model.fittedvalues

plt.figure(figsize=(10, 6))
sns.scatterplot(x=fitted_vals, y=residuals, alpha=0.5, color='steelblue')
plt.axhline(0, color='black', linestyle='--', linewidth=1.5)
plt.title('Residual Diagnostics: Residuals vs. Fitted Values')
plt.xlabel('Fitted Values (Predicted Procedure Cost USD)')
plt.ylabel('Residual Errors')
plt.show()

If the residual plot fans outward as fitted values increase, the model's error variance grows with the size of the prediction, which is a direct violation of the OLS homoscedasticity assumption.

This is the worst-case failure mode for a dynamic pricing algorithm: the model is least reliable exactly where it is most consequential. High-cost procedures receive the most volatile and legally indefensible price recommendations precisely when the financial stakes for both patient and institution are highest. Compounding this, OLS standard errors are artificially compressed under heteroscedasticity, generating false statistical confidence in the most uncertain predictions.

The appropriate remedies are HC3 Heteroscedasticity-Consistent standard errors to correct inference without altering coefficients, or a log-transformation of `Procedure_Cost_USD` to stabilize variance across the prediction range.

## Phase 4: AI Context Engineering (The P.R.I.M.E. Framework)


#### Task 4.1: Lagrange Multiplier Test for Heteroscedasticity

**[Persona]** You are an expert econometrician and Python data scientist specializing in
regression diagnostics and financial risk modeling.

**[Role]** I am a Data Economist at OmniCare Analytics auditing an OLS regression model
predicting Procedure_Cost_USD from patient vitals, clinic capacity, and target-encoded
diagnosis codes built with statsmodels.formula.api.

**[Instructions]** Write a Python script that executes White's Lagrange Multiplier Test for
heteroscedasticity using statsmodels.stats.diagnostic.het_white. The script should:
1. Extract the residuals and design matrix from a fitted statsmodels OLS object called ols_model
2. Pass them into het_white() and unpack all four returned values
3. Print each value labeled clearly as LM Statistic, LM p-value, F Statistic, and F p-value
4. Write a conditional statement that prints a concluding sentence stating whether the null
hypothesis of homoscedasticity is rejected at the 0.05 significance level

**[Meaning]** This test is required to formally confirm heteroscedasticity before applying HC3
robust standard errors. A statistically significant result proves the naive OLS model generates
false confidence and cannot be safely deployed in a live medical pricing environment.

**[Evaluate]** The output should be clean, minimal, and fully executable in Google Colab with no
additional dependencies beyond statsmodels. Include one inline comment per step explaining what
is being extracted and why.

In [ ]:
from statsmodels.stats.diagnostic import het_white

# Extract residuals from the fitted OLS model
residuals = ols_model.resid

# Extract the design matrix (X matrix including the constant column)
exog_matrix = ols_model.model.exog

# Run White's LM Test — returns four values: LM stat, LM p-value, F stat, F p-value
lm_stat, lm_pval, f_stat, f_pval = het_white(residuals, exog_matrix)

# Print all four results with clear labels
print("--- WHITE'S LM TEST FOR HETEROSCEDASTICITY ---")
print(f"LM Statistic:  {lm_stat:.4f}")
print(f"LM p-value:    {lm_pval:.4f}")
print(f"F Statistic:   {f_stat:.4f}")
print(f"F p-value:     {f_pval:.4f}")

# Conclude whether to reject the null hypothesis of homoscedasticity
if lm_pval < 0.05:
    print("\nConclusion: The null hypothesis of homoscedasticity is rejected "
          f"(LM p-value = {lm_pval:.4f} < 0.05). Heteroscedasticity is statistically "
          "confirmed. HC3 robust standard errors are required before deployment.")
else:
    print("\nConclusion: We fail to reject the null hypothesis of homoscedasticity "
          f"(LM p-value = {lm_pval:.4f} > 0.05). No significant evidence of "
          "heteroscedasticity detected.")